# 4.0 諸々試作ノートブック

**目的**: 特徴量エンジニアリング・モデル学習の試作を行う穴あきノートブック

**データ**: `cleaned_three_species_train.csv`（shape: 1167×1559, 13樹種）

**評価指標**: RMSE（小さいほど良い）

**CV戦略**: GroupKFold by 樹種（樹種単位の分割 ← testの樹種はtrainと完全に異なるため）

---

## 全体フロー

```
1. 環境セットアップ
2. データ読み込み・確認
3. 前処理（SNV / SG2次微分 / MSC など）
4. ★ 特徴量エンジニアリング（穴あき）
5. CV設定
6. ★ モデル学習（穴あき）
7. ★ アンサンブル（穴あき）
8. Submission生成
```

## 1. 環境セットアップ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Hiragino Sans'

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error

import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from project.config import RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR

## 2. データ読み込み・確認

In [ ]:
# 学習データ読み込み
train = pd.read_csv(INTERIM_DATA_DIR / 'cleaned_three_species_train.csv', encoding='shift-jis')

# テストデータ読み込み
test = pd.read_csv(RAW_DATA_DIR / 'test.csv', encoding='shift-jis')

print(f'train shape: {train.shape}')
print(f'test shape:  {test.shape}')
print(f'\ntrain 樹種: {train["樹種"].unique()}')
print(f'test  樹種: {test["樹種"].unique()}')

In [ ]:
# メタ列 / スペクトル列 / ターゲット列の分離
META_COLS = ['sample number', 'species number', '樹種']
TARGET_COL = '含水率'
SPEC_COLS = [c for c in train.columns if c not in META_COLS + [TARGET_COL]]

wavenumbers = np.array(SPEC_COLS, dtype=float)  # 波数軸（高波数→低波数）

X_train_raw = train[SPEC_COLS].values
y_train     = train[TARGET_COL].values
groups      = train['樹種'].values         # GroupKFold用

X_test_raw  = test[SPEC_COLS].values

print(f'スペクトル列数: {len(SPEC_COLS)}')
print(f'波数範囲: {wavenumbers.max():.1f} ~ {wavenumbers.min():.1f} cm⁻¹')

## 3. 前処理

| 手法 | 効果 | 実装状況 |
|---|---|---|
| SNV | 行単位の標準化。散乱・光路長変動を補正 | ✅ 実装済み |
| SG 2次微分 | ベースラインのオフセット・傾きを除去 | ✅ 実装済み (window=11, poly=2, deriv=2) |
| MSC | 散乱補正。trainの平均スペクトルを基準にする | ✅ 実装済み |

In [ ]:
def snv(X: np.ndarray) -> np.ndarray:
    """Standard Normal Variate: 行単位で平均0・分散1に正規化"""
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / std


def savgol_derivative(X: np.ndarray, window: int = 11, poly: int = 2, deriv: int = 2) -> np.ndarray:
    """Savitzky-Golay フィルタによる微分"""
    from scipy.signal import savgol_filter
    return savgol_filter(X, window_length=window, polyorder=poly, deriv=deriv, axis=1)


def msc(X: np.ndarray, ref: np.ndarray = None) -> tuple:
    """
    Multiplicative Scatter Correction
    ref: 基準スペクトル（Noneの場合はXの平均スペクトルを使用）
    ※ testに適用する際は train の ref_spectrum を渡すこと
    戻り値: (補正済みスペクトル, 使用した基準スペクトル)
    """
    if ref is None:
        ref = X.mean(axis=0)
    X_msc = np.zeros_like(X)
    for i in range(X.shape[0]):
        coef = np.polyfit(ref, X[i], 1)
        X_msc[i] = (X[i] - coef[1]) / coef[0]
    return X_msc, ref


print('前処理関数の定義完了')

In [ ]:
# --- 前処理の適用 ---

# SNV
X_train_snv = snv(X_train_raw)
X_test_snv  = snv(X_test_raw)

# SNV + SG2次微分
X_train_sg = savgol_derivative(X_train_snv, window=11, poly=2, deriv=2)
X_test_sg  = savgol_derivative(X_test_snv,  window=11, poly=2, deriv=2)

# MSC（trainの平均スペクトルを基準にしてtestにも適用）
X_train_msc, ref_spectrum = msc(X_train_raw)
X_test_msc,  _            = msc(X_test_raw, ref=ref_spectrum)

print('各前処理済みスペクトル shape:')
print(f'  SNV:       {X_train_snv.shape}')
print(f'  SNV+SG2d:  {X_train_sg.shape}')
print(f'  MSC:       {X_train_msc.shape}')

## 4. ★ 特徴量エンジニアリング（穴あき）

**ここで特徴量エンジニアリングを行います。**

以下の候補から選択・実装してください。

| 特徴量アイデア | 概要 | 難易度 |
|---|---|---|
| **水の吸収ピーク帯域抽出** | 5200 / 6900 / 8500 cm⁻¹ 周辺の吸光度・面積 | ★☆☆ |
| **PCA次元削減** | SNV後スペクトルをPCAで圧縮 | ★☆☆ |
| **スペクトル比・差** | 特定波数間の比または差分 | ★☆☆ |
| **統計量特徴量** | 各帯域の平均・標準偏差・積分値など | ★☆☆ |
| **樹種名のテキストembedding** | sentence-transformers で樹種名をベクトル化 | ★★☆ |
| **木材画像のCLIP embedding** | train13樹種の画像のみ使用可（testの6樹種はNG） | ★★★ |

In [ ]:
# ===============================================================
# 4-A: 水の吸収ピーク帯域の特徴量抽出
# 対象波数帯: ~8500 cm⁻¹（OHの第2倍音）/ ~6900 cm⁻¹（第1倍音）/ ~5200 cm⁻¹（結合音）
# ===============================================================

def extract_peak_features(X: np.ndarray, wavenumbers: np.ndarray) -> pd.DataFrame:
    """
    水の吸収ピーク帯域周辺の特徴量を抽出する
    
    TODO: 以下を実装してください
      - 各ピーク帯域（例: 8400~8600, 6800~7000, 5100~5300 cm⁻¹）の範囲を定義
      - 各帯域の平均吸光度、最大値、積分値（台形則）などを特徴量として計算
      - 戻り値は pd.DataFrame（行=サンプル、列=特徴量名）
    """
    features = {}
    
    # --- ここを実装 ---
    # 例:
    # peak_bands = {
    #     'OH2nd': (8400, 8600),
    #     'OH1st': (6800, 7000),
    #     'comb':  (5100, 5300),
    # }
    # for name, (low, high) in peak_bands.items():
    #     mask = (wavenumbers >= low) & (wavenumbers <= high)
    #     features[f'{name}_mean'] = X[:, mask].mean(axis=1)
    #     features[f'{name}_max']  = X[:, mask].max(axis=1)
    #     features[f'{name}_area'] = np.trapz(X[:, mask], wavenumbers[mask], axis=1)
    # -----------------
    
    return pd.DataFrame(features)


feat_peak_train = extract_peak_features(X_train_snv, wavenumbers)
feat_peak_test  = extract_peak_features(X_test_snv, wavenumbers)
print('ピーク特徴量 shape:', feat_peak_train.shape)
print(feat_peak_train.head(2))

In [ ]:
# ===============================================================
# 4-B: PCA次元削減
# ===============================================================
from sklearn.decomposition import PCA

def fit_pca(X_train: np.ndarray, n_components: int = 50):
    """
    trainでPCAをfitし、変換済み特徴量を返す
    
    TODO: 以下を実装してください
      - StandardScalerでスケーリング後にPCAを適用
      - 累積寄与率を確認して適切なn_componentsを選ぶ
      - scaler と pca を返して test にも同じ変換を適用できるようにする
    """
    # --- ここを実装 ---
    scaler = None  # StandardScaler()
    pca    = None  # PCA(n_components=n_components)
    X_pca  = None  # 変換後の配列
    # -----------------
    
    return scaler, pca, X_pca


# 使用例:
# scaler_pca, pca_model, X_train_pca = fit_pca(X_train_snv, n_components=50)
# X_test_pca = pca_model.transform(scaler_pca.transform(X_test_snv))
# print('PCA特徴量 shape:', X_train_pca.shape)

In [ ]:
# ===============================================================
# 4-C: 特徴量の結合
# 上で作った特徴量を numpy 配列にまとめる
# ===============================================================

# TODO: 使う特徴量を選択してここで結合してください
# 例（PCA + ピーク特徴量）:
# X_train_feat = np.hstack([X_train_pca, feat_peak_train.values])
# X_test_feat  = np.hstack([X_test_pca,  feat_peak_test.values])

# デフォルト: SNV 生スペクトルをそのまま使用
X_train_feat = X_train_snv
X_test_feat  = X_test_snv

print('最終特徴量 shape（train）:', X_train_feat.shape)
print('最終特徴量 shape（test）: ', X_test_feat.shape)

## 5. CV設定

樹種単位のGroupKFoldを使用（testの樹種はtrainと完全に異なるため）

In [ ]:
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

print(f'GroupKFold（n_splits={N_SPLITS}、グループ=樹種）')
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_feat, y_train, groups=groups)):
    tr_species = set(groups[tr_idx])
    va_species = set(groups[va_idx])
    print(f'  Fold {fold}: val={sorted(va_species)}')

## 6. ★ モデル学習（穴あき）

**ここでモデル学習を行います。**

以下のモデルから選択・実装してください。

| モデル | 特徴 | 推奨前処理 |
|---|---|---|
| **PLSR** | NIRに強い。高次元スペクトルを直接入力可 | SNV+SG2次微分 |
| **LightGBM** | 特徴量の選択・非線形関係に強い | SNV+PCA |
| **Ridge回帰** | シンプルで解釈しやすい | SNV+PCA/SG |
| **SVR** | 小サンプルに強い | SNV+SG |

In [ ]:
# ===============================================================
# 6-A: PLSR
# ===============================================================

def train_plsr_cv(X: np.ndarray, y: np.ndarray, groups: np.ndarray,
                  X_test: np.ndarray,
                  n_components_range=range(2, 30)) -> dict:
    """
    PLSRのn_componentsをCVで探索し最適値でOOF予測を返す
    
    TODO: 以下を実装してください
      1. n_components_range の各値でGroupKFold CVを実施
      2. 各n_componentsのCV RMSEを記録
      3. 最適なn_componentsで再度CVしてOOF予測を生成
      4. 全trainデータでfitしてtest予測を返す
    """
    best_n    = None
    best_rmse = np.inf
    oof_preds = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    
    # --- ここを実装 ---
    
    # -----------------
    
    print(f'PLSR 最適 n_components: {best_n}, CV RMSE: {best_rmse:.4f}')
    return {
        'model_name': 'PLSR',
        'oof_preds':  oof_preds,
        'test_preds': test_preds,
        'cv_rmse':    best_rmse,
        'best_params': {'n_components': best_n},
    }


# 実行する場合はコメントアウトを解除
# result_plsr = train_plsr_cv(X_train_sg, y_train, groups, X_test=X_test_sg)

In [ ]:
# ===============================================================
# 6-B: LightGBM
# ===============================================================

def train_lgbm_cv(X: np.ndarray, y: np.ndarray, groups: np.ndarray,
                  X_test: np.ndarray,
                  params: dict = None) -> dict:
    """
    LightGBMをGroupKFoldでCV学習し、OOF予測とtest予測を返す
    
    TODO: 以下を実装してください
      1. params が None の場合のデフォルトパラメータを設定
         （objective='regression', metric='rmse'）
      2. GroupKFoldで各foldを学習
      3. Early stoppingを使用してoverfittingを防ぐ
      4. OOF予測 / test予測（各foldの平均）を返す
    """
    if params is None:
        params = {
            'objective':        'regression',
            'metric':           'rmse',
            'learning_rate':    0.05,
            'num_leaves':       31,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq':     5,
            'random_state':     42,
            'verbose':          -1,
        }
    
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    
    # --- ここを実装 ---
    
    # -----------------
    
    cv_rmse = root_mean_squared_error(y, oof_preds)
    print(f'LightGBM CV RMSE: {cv_rmse:.4f}')
    return {
        'model_name': 'LightGBM',
        'oof_preds':  oof_preds,
        'test_preds': test_preds,
        'cv_rmse':    cv_rmse,
    }


# 実行する場合はコメントアウトを解除
# result_lgbm = train_lgbm_cv(X_train_feat, y_train, groups, X_test=X_test_feat)

## 7. ★ アンサンブル（穴あき）

**ここで複数モデルの予測を組み合わせます。**

方法:
- **単純平均**: 各モデルのtest予測を等重みで平均
- **加重平均**: OOF RMSEをもとに重みを決定（RMSEが低いモデルを重視）
- **Stacking**: OOF予測をmeta-learnerの入力とする

In [ ]:
# ===============================================================
# 7-A: アンサンブル
# ===============================================================

def ensemble_predictions(results: list, weights: list = None) -> np.ndarray:
    """
    複数モデルのtest予測をアンサンブルする
    
    Args:
        results: train_*_cv の返り値のリスト
        weights: 各モデルの重み（None の場合は等重み平均）
    
    TODO: 以下を実装してください
      - weights が None なら各モデルを等重みで平均
      - OOF RMSEベースの自動重み付け（任意）: 1/RMSE を正規化して使う
    """
    preds = np.stack([r['test_preds'] for r in results], axis=0)  # shape: (n_models, n_test)
    
    # --- ここを実装 ---
    pred = None
    # -----------------
    
    return pred


# 使用例（2つ以上のモデルを学習した後に実行）:
# results = [result_plsr, result_lgbm]
# test_pred_ensemble = ensemble_predictions(results)
# print('Ensemble test_preds shape:', test_pred_ensemble.shape)

## 8. Submission生成

In [ ]:
def save_submission(test_preds: np.ndarray, version: str = 'v4_0') -> None:
    """
    submission.csv を保存する
    フォーマット: header=False, encoding=shift-jis（sample_submit.csv に準拠）
    """
    sample_submit = pd.read_csv(RAW_DATA_DIR / 'sample_submit.csv',
                                encoding='shift-jis', header=None)
    submission = sample_submit.copy()
    submission.iloc[:, 1] = test_preds
    
    save_path = PROCESSED_DATA_DIR / f'submission_{version}.csv'
    submission.to_csv(save_path, index=False, header=False, encoding='shift-jis')
    print(f'Saved: {save_path}')
    print(submission.iloc[:, 1].describe())


# 使用例:
# save_submission(test_pred_ensemble, version='v4_0_ensemble')

## 9. 結果まとめ

モデルを試したら以下のテーブルを更新してください。

| version | 前処理 | 特徴量 | モデル | CV RMSE | LB スコア | 備考 |
|---|---|---|---|---|---|---|
| v4_0_baseline | SNV | 生スペクトル | - | - | - | 穴あきノートブック初期状態 |